In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().resolve()
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

def zscore(s):
    sd = s.std(ddof=0)
    return (s - s.mean()) / sd if pd.notna(sd) and sd != 0 else np.nan

def rank_desc(s):
    return s.rank(ascending=False, method="first")

def rank_asc(s):
    return s.rank(ascending=True, method="first")

def build_portfolio(df, score_col, date_col, ticker_col="ticker", top_n=20, long_only=True):
    x = df.copy()
    x = x.dropna(subset=[score_col])
    x["rank"] = x.groupby(date_col)[score_col].transform(rank_desc)
    x["pct_rank"] = x.groupby(date_col)[score_col].transform(lambda s: s.rank(pct=True, ascending=False))
    x["selected"] = x["rank"] <= top_n
    if long_only:
        x["weight"] = np.where(x["selected"], 1.0 / top_n, 0.0)
    else:
        x["weight"] = np.where(x["selected"], 1.0 / top_n, 0.0)
        x["weight"] = x["weight"] / x.groupby(date_col)["weight"].transform(lambda s: s.abs().sum() if s.abs().sum() != 0 else np.nan)
    return x

#load model outputs
a_value = pd.read_csv(ROOT / "output" / "factor_A_value_358.csv")
a_quality = pd.read_csv(ROOT / "output" / "factor_A_quality_358.csv")
a_sue = pd.read_csv(ROOT / "output" / "factor_A_sue_358.csv")
a_mom = pd.read_csv(ROOT / "output" / "factor_A_momentum_358.csv", parse_dates=["date"])

b_price = pd.read_csv(ROOT / "output" / "factor_B_price_only_500.csv", parse_dates=["date"])
c_value = pd.read_csv(ROOT / "output" / "factor_C_value_overlap.csv")
c_quality = pd.read_csv(ROOT / "output" / "factor_C_quality_overlap.csv")
c_sue = pd.read_csv(ROOT / "output" / "factor_C_sue_overlap.csv")
c_mom = pd.read_csv(ROOT / "output" / "factor_C_momentum_overlap.csv", parse_dates=["date"])

#standardize column names
for df in [a_value, a_quality, a_sue, c_value, c_quality, c_sue]:
    df["ticker"] = df["ticker"].astype(str)
    df["report_date"] = pd.to_datetime(df["report_date"])

a_mom["ticker"] = a_mom["ticker"].astype(str)
b_price["ticker"] = b_price["ticker"].astype(str)
c_mom["ticker"] = c_mom["ticker"].astype(str)

#model a: core 358
a_core = (
    a_value[["ticker", "report_date", "value_score"]]
    .merge(a_quality[["ticker", "report_date", "quality_score"]], on=["ticker", "report_date"], how="inner")
    .merge(a_sue[["ticker", "report_date", "sue_z"]], on=["ticker", "report_date"], how="inner")
)

a_mom["report_date"] = a_mom["date"]
a_core = a_core.merge(
    a_mom[["ticker", "report_date", "momentum_12m_z"]],
    on=["ticker", "report_date"],
    how="inner"
)

a_core["combined_score"] = a_core[["value_score", "quality_score", "sue_z", "momentum_12m_z"]].mean(axis=1)
a_port = build_portfolio(a_core, "combined_score", "report_date", top_n=20, long_only=True)
a_port.to_csv(OUT / "portfolio_A_core_358.csv", index=False)

#model b: price only 500
b = b_price.rename(columns={"date": "report_date"}).copy()
b["combined_score"] = b["momentum_12m_z"]
b_port = build_portfolio(b, "combined_score", "report_date", top_n=50, long_only=True)
b_port.to_csv(OUT / "portfolio_B_price_only_500.csv", index=False)

#model c: price plus fundamentals overlap
c_core = (
    c_value[["ticker", "report_date", "value_score"]]
    .merge(c_quality[["ticker", "report_date", "quality_score"]], on=["ticker", "report_date"], how="inner")
    .merge(c_sue[["ticker", "report_date", "sue_z"]], on=["ticker", "report_date"], how="inner")
)
c_mom["report_date"] = c_mom["date"]
c_core = c_core.merge(
    c_mom[["ticker", "report_date", "momentum_12m_z"]],
    on=["ticker", "report_date"],
    how="inner"
)

c_core["combined_score"] = c_core[["value_score", "quality_score", "sue_z", "momentum_12m_z"]].mean(axis=1)
c_port = build_portfolio(c_core, "combined_score", "report_date", top_n=30, long_only=True)
c_port.to_csv(OUT / "portfolio_C_overlap.csv", index=False)

#coverage summaries
cov = pd.DataFrame([
    {"model": "A_core_358", "n_rows": len(a_core), "n_tickers": a_core["ticker"].nunique(), "top_n": 20},
    {"model": "B_price_only_500", "n_rows": len(b), "n_tickers": b["ticker"].nunique(), "top_n": 50},
    {"model": "C_overlap", "n_rows": len(c_core), "n_tickers": c_core["ticker"].nunique(), "top_n": 30},
])
cov.to_csv(OUT / "portfolio_coverage.csv", index=False)

print(cov)
print(a_port.head())
print(b_port.head())
print(c_port.head())

              model  n_rows  n_tickers  top_n
0        A_core_358    6103        327     20
1  B_price_only_500   65148        498     50
2         C_overlap    6148        357     30
  ticker report_date  value_score  quality_score     sue_z  momentum_12m_z  \
0      A  2020-07-31     0.679565       0.195519       NaN        0.249283   
1      A  2020-10-31     0.512353       0.046085       NaN        0.200017   
2      A  2021-01-31     0.329747      -0.145433       NaN        0.205848   
3      A  2021-04-30     0.385610      -0.033535       NaN       -0.034373   
4      A  2021-07-31     0.341430       0.089155 -0.267273        0.368246   

   combined_score  rank  pct_rank  selected  weight  
0        0.374789  11.0  0.305556      True    0.05  
1        0.252818  10.0  0.277778      True    0.05  
2        0.130054  11.0  0.297297      True    0.05  
3        0.105901  14.0  0.378378      True    0.05  
4        0.132890  13.0  0.351351      True    0.05  
  report_date ticker  m